<a href="https://colab.research.google.com/github/HVDEER/google_colab/blob/main/creditcard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving CreditCardApprovalPredictionMoneyMan.zip to CreditCardApprovalPredictionMoneyMan.zip


In [2]:
!unzip -q CreditCardApprovalPredictionMoneyMan

In [3]:
!ls

application_record.csv			  credit_record.csv
CreditCardApprovalPredictionMoneyMan.zip  sample_data


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

application_df = pd.read_csv("application_record.csv")
credit_df = pd.read_csv("credit_record.csv")

print("Application Record Shape:", application_df.shape)
print("Credit Record Shape:", credit_df.shape)


Application Record Shape: (438557, 18)
Credit Record Shape: (1048575, 3)


In [5]:
application_df.info()
credit_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438557 entries, 0 to 438556
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ID                   438557 non-null  int64  
 1   CODE_GENDER          438557 non-null  object 
 2   FLAG_OWN_CAR         438557 non-null  object 
 3   FLAG_OWN_REALTY      438557 non-null  object 
 4   CNT_CHILDREN         438557 non-null  int64  
 5   AMT_INCOME_TOTAL     438557 non-null  float64
 6   NAME_INCOME_TYPE     438557 non-null  object 
 7   NAME_EDUCATION_TYPE  438557 non-null  object 
 8   NAME_FAMILY_STATUS   438557 non-null  object 
 9   NAME_HOUSING_TYPE    438557 non-null  object 
 10  DAYS_BIRTH           438557 non-null  int64  
 11  DAYS_EMPLOYED        438557 non-null  int64  
 12  FLAG_MOBIL           438557 non-null  int64  
 13  FLAG_WORK_PHONE      438557 non-null  int64  
 14  FLAG_PHONE           438557 non-null  int64  
 15  FLAG_EMAIL       

In [6]:
# Check unique identifiers
print("Unique IDs in application_record:", application_df['ID'].nunique())
print("Unique IDs in credit_record:", credit_df['ID'].nunique())


Unique IDs in application_record: 438510
Unique IDs in credit_record: 45985


In [7]:
# Credit status distribution (raw behavioral signal)
credit_df['STATUS'].value_counts().sort_index()


,count
STATUS,
0,383120
1,11090
2,868
3,320
4,223
5,1693
C,442031
X,209230


an integer (whole number) stored using 64 bits of memory.

In [8]:
application_df.head()


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


In [9]:
application_df.describe()


,ID,CNT_CHILDREN,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS
count,4.385570e+05,438557.000000,4.385570e+05,438557.000000,438557.000000,438557.0,438557.000000,438557.000000,438557.000000,438557.000000
mean,6.022176e+06,0.427390,1.875243e+05,-15997.904649,60563.675328,1.0,0.206133,0.287771,0.108207,2.194465
std,5.716370e+05,0.724882,1.100869e+05,4185.030007,138767.799647,0.0,0.404527,0.452724,0.310642,0.897207
min,5.008804e+06,0.000000,2.610000e+04,-25201.000000,-17531.000000,1.0,0.000000,0.000000,0.000000,1.000000
25%,5.609375e+06,0.000000,1.215000e+05,-19483.000000,-3103.000000,1.0,0.000000,0.000000,0.000000,2.000000
50%,6.047745e+06,0.000000,1.607805e+05,-15630.000000,-1467.000000,1.0,0.000000,0.000000,0.000000,2.000000
75%,6.456971e+06,1.000000,2.250000e+05,-12514.000000,-371.000000,1.0,0.000000,1.000000,0.000000,3.000000
max,7.999952e+06,19.000000,6.750000e+06,-7489.000000,365243.000000,1.0,1.000000,1.000000,1.000000,20.000000


What does 4.24234e+05 mean?

In [10]:
credit_df.head()


,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C


In [11]:
credit_df.describe()

,ID,MONTHS_BALANCE
count,1.048575e+06,1.048575e+06
mean,5.068286e+06,-1.913700e+01
std,4.615058e+04,1.402350e+01
min,5.001711e+06,-6.000000e+01
25%,5.023644e+06,-2.900000e+01
50%,5.062104e+06,-1.700000e+01
75%,5.113856e+06,-7.000000e+00
max,5.150487e+06,0.000000e+00


In [12]:
# Map STATUS to risk severity (higher = worse)
status_severity = {
    'C': 0,   # paid off
    'X': 0,   # no loan (neutral)
    '0': 1,   # 1–29 days late
    '1': 2,   # 30–59 days late
    '2': 3,   # 60–89 days late
    '3': 4,   # 90–119 days late
    '4': 5,   # 120–149 days late
    '5': 6    # 150+ days / write-off
}

credit_df['STATUS_SEVERITY'] = credit_df['STATUS'].map(status_severity)


In [13]:
credit_agg = (
    credit_df
    .groupby('ID')
    .agg(
        worst_status=('STATUS_SEVERITY', 'max'),
        total_months=('STATUS_SEVERITY', 'count'),
        delinquent_months=('STATUS_SEVERITY', lambda x: (x >= 2).sum()),
        severe_delinquency_months=('STATUS_SEVERITY', lambda x: (x >= 4).sum())
    )
    .reset_index()
)


In [14]:
credit_agg['ever_delinquent'] = credit_agg['delinquent_months'] > 0
credit_agg['ever_severely_delinquent'] = credit_agg['severe_delinquency_months'] > 0
credit_agg['ever_defaulted'] = credit_agg['worst_status'] >= 6


In [15]:
credit_agg.head()


,ID,worst_status,total_months,delinquent_months,severe_delinquency_months,ever_delinquent,ever_severely_delinquent,ever_defaulted
0,5001711,1,4,0,0,False,False,False
1,5001712,1,19,0,0,False,False,False
2,5001713,0,22,0,0,False,False,False
3,5001714,0,15,0,0,False,False,False
4,5001715,0,60,0,0,False,False,False


In [16]:
decision_df = application_df.merge(
    credit_agg,
    on='ID',
    how='left'
)


In [17]:
decision_df.shape


(438557, 25)

In [18]:
decision_df[['worst_status', 'ever_defaulted']].isna().mean()


,0
worst_status,0.916871
ever_defaulted,0.916871


In [19]:
decision_df['risk_outcome'] = np.select(
    [
        decision_df['ever_defaulted'] == True,
        decision_df['worst_status'] >= 4,
        decision_df['worst_status'] >= 2
    ],
    [
        'Defaulted',
        'High Risk',
        'Moderate Risk'
    ],
    default='Low / No Observed Risk'
)


In [20]:
decision_df['risk_outcome'].value_counts(normalize=True)


,proportion
risk_outcome,
Low / No Observed Risk,0.990216
Moderate Risk,0.009096
Defaulted,0.000410
High Risk,0.000278


In [21]:
decision_df.groupby('risk_outcome')['AMT_INCOME_TOTAL'].describe()


,count,mean,std,min,25%,50%,75%,max
risk_outcome,,,,,,,,
Defaulted,180.0,200392.500000,114851.755492,65250.0,126000.0,175500.0,225000.0,675000.0
High Risk,122.0,177997.131148,82248.154115,36000.0,112500.0,180000.0,225000.0,459000.0
Low / No Observed Risk,434266.0,187465.927400,110020.814949,26100.0,121500.0,160200.0,225000.0,6750000.0
Moderate Risk,3989.0,193588.260466,117379.686682,27000.0,126000.0,171000.0,225000.0,1575000.0


In [22]:
decision_df.groupby('risk_outcome')['DAYS_EMPLOYED'].describe()


,count,mean,std,min,25%,50%,75%,max
risk_outcome,,,,,,,,
Defaulted,180.0,60966.105556,139193.227979,-12827.0,-2485.25,-1081.0,-499.0,365243.0
High Risk,122.0,97472.032787,163730.367295,-11398.0,-1890.75,-796.0,365243.0,365243.0
Low / No Observed Risk,434266.0,60656.836089,138848.887720,-17531.0,-3102.00,-1466.0,-370.0,365243.0
Moderate Risk,3989.0,49274.678616,128121.583043,-15038.0,-3246.00,-1600.0,-508.0,365243.0


In [23]:
pd.crosstab(
    decision_df['CODE_GENDER'],
    decision_df['risk_outcome'],
    normalize='index'
)


risk_outcome,Defaulted,High Risk,Low / No Observed Risk,Moderate Risk
CODE_GENDER,,,,
F,0.000377,0.000255,0.990633,0.008735
M,0.000479,0.000326,0.989363,0.009832


In [24]:
pd.crosstab(
    decision_df['NAME_EDUCATION_TYPE'],
    decision_df['risk_outcome'],
    normalize='index'
)


risk_outcome,Defaulted,High Risk,Low / No Observed Risk,Moderate Risk
NAME_EDUCATION_TYPE,,,,
Academic degree,0.000000,0.000000,0.977564,0.022436
Higher education,0.000485,0.000289,0.990232,0.008994
Incomplete higher,0.000471,0.000875,0.986062,0.012592
Lower secondary,0.001481,0.000247,0.990373,0.007899
Secondary / secondary special,0.000364,0.000245,0.990425,0.008966


In [25]:
# Flag bad statuses
bad_statuses = ['3', '4', '5']

credit_df['is_bad'] = credit_df['STATUS'].isin(bad_statuses)

# Aggregate to customer level
credit_agg = credit_df.groupby('ID').agg(
    total_months=('STATUS', 'count'),
    bad_months=('is_bad', 'sum')
).reset_index()

credit_agg.head()


,ID,total_months,bad_months
0,5001711,4,0
1,5001712,19,0
2,5001713,22,0
3,5001714,15,0
4,5001715,60,0


In [26]:
credit_agg['risk_label'] = credit_agg['bad_months'].apply(
    lambda x: 'High Risk' if x > 0 else 'Low Risk'
)

credit_agg['risk_label'].value_counts()


,count
risk_label,
Low Risk,45654
High Risk,331


In [27]:
full_df = application_df.merge(credit_agg, on='ID', how='inner')

full_df.head()


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,...,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,total_months,bad_months,risk_label
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,-4542,1,1,0,0,NaN,2.0,16,0,Low Risk
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,-4542,1,1,0,0,NaN,2.0,15,0,Low Risk
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,...,-1134,1,0,0,0,Security staff,2.0,30,0,Low Risk
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,-3051,1,0,1,1,Sales staff,1.0,5,0,Low Risk
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,-3051,1,0,1,1,Sales staff,1.0,5,0,Low Risk


In [28]:
bad_statuses = ['3', '4', '5']

credit_df['is_bad'] = credit_df['STATUS'].isin(bad_statuses)


In [29]:
credit_agg = credit_df.groupby('ID').agg(
    total_months=('STATUS', 'count'),
    bad_months=('is_bad', 'sum')
).reset_index()


In [30]:
credit_agg['risk_label'] = credit_agg['bad_months'].apply(
    lambda x: 'High Risk' if x > 0 else 'Low Risk'
)


In [31]:
high_risk = full_df[full_df['risk_label'] == 'High Risk']
low_risk  = full_df[full_df['risk_label'] == 'Low Risk']

high_risk.shape, low_risk.shape


((302, 21), (36155, 21))

In [32]:
numeric_cols = [
    'AMT_INCOME_TOTAL',
    'DAYS_BIRTH',
    'DAYS_EMPLOYED',
    'CNT_FAM_MEMBERS'
]

comparison = full_df.groupby('risk_label')[numeric_cols].median()
comparison


,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,CNT_FAM_MEMBERS
risk_label,,,,
High Risk,180000.0,-16413.0,-925.5,2.0
Low Risk,157500.0,-15560.0,-1557.0,2.0


In [33]:
full_df['AGE_YEARS'] = (-full_df['DAYS_BIRTH'] / 365).round(1)
full_df['EMPLOYED_YEARS'] = (-full_df['DAYS_EMPLOYED'] / 365).round(1)


In [34]:
full_df.groupby('risk_label')[['AGE_YEARS', 'EMPLOYED_YEARS']].median()


,AGE_YEARS,EMPLOYED_YEARS
risk_label,,
High Risk,45.0,2.5
Low Risk,42.6,4.3


In [35]:
education_dist = (
    full_df
    .groupby(['risk_label', 'NAME_EDUCATION_TYPE'])
    .size()
    .groupby(level=0)
    .apply(lambda x: x / x.sum() * 100)
)

education_dist


risk_label  risk_label  NAME_EDUCATION_TYPE          
High Risk   High Risk   Higher education                 30.132450
                        Incomplete higher                 6.622517
                        Lower secondary                   2.317881
                        Secondary / secondary special    60.927152
Low Risk    Low Risk    Academic degree                   0.088508
                        Higher education                 27.030839
                        Incomplete higher                 3.844558
                        Lower secondary                   1.015074
                        Secondary / secondary special    68.021021
dtype: float64

In [36]:
full_df['AGE_GROUP'] = pd.cut(
    full_df['AGE_YEARS'],
    bins=[18, 25, 35, 45, 55, 65, 100],
    labels=['18–25', '26–35', '36–45', '46–55', '56–65', '65+']
)


In [37]:
age_risk = (
    full_df
    .groupby('AGE_GROUP')['risk_label']
    .apply(lambda x: (x == 'High Risk').mean() * 100)
)

age_risk


/tmp/ipython-input-1102998771.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('AGE_GROUP')['risk_label']


,risk_label
AGE_GROUP,
18–25,0.263158
26–35,0.816151
36–45,0.737407
46–55,0.916767
56–65,0.854330
65+,1.606426


In [38]:
gender_risk = (
    full_df
    .groupby('CODE_GENDER')['risk_label']
    .apply(lambda x: (x == 'High Risk').mean() * 100)
)

gender_risk


,risk_label
CODE_GENDER,
F,0.761359
M,0.964497


In [39]:
family_risk = (
    full_df
    .groupby('NAME_FAMILY_STATUS')['risk_label']
    .apply(lambda x: (x == 'High Risk').mean() * 100)
    .sort_values(ascending=False)
)

family_risk


,risk_label
NAME_FAMILY_STATUS,
Separated,1.283880
Widow,1.174935
Single / not married,1.056119
Married,0.790482
Civil marriage,0.271647


In [40]:
full_df.groupby('risk_label')['AMT_INCOME_TOTAL'].median()


,AMT_INCOME_TOTAL
risk_label,
High Risk,180000.0
Low Risk,157500.0


In [41]:
full_df['rule_1_reject'] = (
    (full_df['AMT_INCOME_TOTAL'] < 100000) &
    (full_df['EMPLOYED_YEARS'] < 1)
)


In [42]:
rule1_summary = full_df.groupby('rule_1_reject')['risk_label'].value_counts(normalize=True) * 100
rule1_summary


rule_1_reject  risk_label
False          Low Risk      99.186567
               High Risk      0.813433
True           Low Risk      98.918919
               High Risk      1.081081
Name: proportion, dtype: float64

In [43]:
full_df['rule_2_reject'] = (
    (full_df['AGE_YEARS'] < 25) &
    (full_df['EMPLOYED_YEARS'] < 0.5)
)


In [44]:
full_df.groupby('rule_2_reject')['risk_label'].value_counts(normalize=True) * 100


rule_2_reject  risk_label
False          Low Risk       99.170512
               High Risk       0.829488
True           Low Risk      100.000000
Name: proportion, dtype: float64

In [45]:
rules_eval = pd.DataFrame({
    'Rule 1 Reject Rate': full_df['rule_1_reject'].mean() * 100,
    'Rule 2 Reject Rate': full_df['rule_2_reject'].mean() * 100
}, index=['% Applicants Rejected'])

rules_eval


,Rule 1 Reject Rate,Rule 2 Reject Rate
% Applicants Rejected,5.581918,0.134405


In [46]:
from sklearn.linear_model import LogisticRegression

features = [
    'AMT_INCOME_TOTAL',
    'AGE_YEARS',
    'EMPLOYED_YEARS',
    'CNT_FAM_MEMBERS'
]

X = full_df[features]
y = (full_df['risk_label'] == 'High Risk').astype(int)


In [47]:
model = LogisticRegression(max_iter=1000)
model.fit(X, y)


LogisticRegression(max_iter=1000)

In [48]:
importance = pd.Series(
    model.coef_[0],
    index=features
).sort_values()

importance


,0
CNT_FAM_MEMBERS,-9.741530e-02
EMPLOYED_YEARS,-2.286039e-04
AMT_INCOME_TOTAL,6.109645e-07
AGE_YEARS,2.251203e-03


In [49]:
# Final policy evaluation using Rule 1 as example

policy_reject = full_df['rule_1_reject']

summary = pd.crosstab(
    policy_reject,
    full_df['risk_label'],
    normalize='index'
) * 100

summary


risk_label,High Risk,Low Risk
rule_1_reject,,
False,0.813433,99.186567
True,1.081081,98.918919


In [50]:
from sklearn.metrics import accuracy_score
accuracy_score(y, model.predict(X))


0.9917162684806758